# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token='{HF_TOKEN}';")

# Mid-panel month for ALL contract work. Never the _sample table for label logic:
# _sample IS the final month (June 2026) -- it's the sealed test/outcome window, not a random sample.
MONTH = "2026-03"
FACT = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
print("Using mid-panel month:", MONTH)
print("Path (adjust if the actual partition layout differs -- check the dataset file listing first):")
print(FACT)

HF_TOKEN: ··········


CatalogException: Catalog Error: unrecognized configuration parameter "hf_token"

Did you mean: "http_keep_alive"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. One row means:**
One page, on one specific day, for one client. The raw data is basically: "On March 3rd, this page (for this client) got X views and Y clicks." I group many of these daily rows together into two chunks per page — a "before" chunk and an "after" chunk — to make my prediction and check it.

**2. Which table(s):**
The daily performance table (`fact_content_daily_performance`), but only the March 2026 slice. I'm using one single month to start, not the whole multi-year dataset — that comes later once this small version works. I'm specifically avoiding the "sample" table, because that one happens to *be* the very last month of data (June 2026), which I need to save untouched for a final, honest test later — using it now would be like peeking at the exam before studying.

**3. Time window:**
I pick a pretend "today" — March 15, 2026. Everything from March 1–14 is the "before" period (what I'm allowed to look at). Everything from March 15–31 is the "after" period (what actually happened next, used only to check the answer).

**4. What I'm trying to predict:**
Whether a page's traffic dropped a lot — specifically, if its average daily views in the "after" period are more than 20% lower than in the "before" period, I call that page "declining." That's the thing I'm trying to guess ahead of time.

**5. What I deliberately leave out:**
I never let my prediction use any information from the "after" period — because that's literally what I'm trying to guess, not something I should already know. If I accidentally included "after" numbers as an input, my model would look amazing but would actually just be cheating by reading the answer key. That's the trap the rest of the notebook deliberately demonstrates and then fixes.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# QUERY 2 -- ROW COUNT + DATE SPAN for my lane's slice (month = 2026-03).
count_span = con.execute(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM '{FACT}'
""").df()
print("Row count + date span for month=2026-03:")
count_span

NameError: name 'con' is not defined

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.